# 03 — Alert Postmortem

**Was Du hier siehst:** Wie haben sich Instrumente *nach* einem Monitor-Alert entwickelt? Für jedes Alert der letzten 30 Tage werden Forward-Returns über +1d / +5d / +20d berechnet — auf Basis `mkt_quotes_daily`-`close`. Pro Regel + Richtung wird die Hit-Rate (Anteil mit positivem Return) gegen die Direction-Erwartung gemessen.

**Datenquellen:**
- `sig_alerts` — alle Alerts mit `rule_name`, `direction`, `trigger_value`, `threshold`
- `sig_alert_explanations` — LLM-Kontext (`sentiment`, `news_used`, `confidence`)
- `mkt_quotes_daily` — `close` für Forward-Window-Berechnung

**Fragen die Du damit beantworten kannst:**
- Welche Monitor-Regel hat die beste Vorhersagekraft (in welche Richtung)?
- Korreliert das LLM-`sentiment` mit dem tatsächlichen Forward-Return?
- Sind Alerts mit `news_used > 0` qualitativ anders als ohne?

**Limitations:** Kleines N pro Regel; +20d kann für sehr junge Alerts noch nicht abgeschlossen sein (NaN sind erwartet). Keine Risk-Adjustierung.

In [ ]:
import sys, pathlib
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
import numpy as np
from datetime import date, timedelta

from modules.analysis._db import session, df
from modules.analysis._plot import setup, COLORS, hline, plt
setup()

In [ ]:
LOOKBACK_DAYS    = 30          # Alert-Zeitfenster
FORWARD_WINDOWS  = [1, 5, 20]  # Trading-Tage forward
MIN_ALERTS_PER_RULE = 3        # Hit-Rate erst ab N Beobachtungen anzeigen

## 1. Alerts + LLM-Kontext laden

In [ ]:
since = date.today() - timedelta(days=LOOKBACK_DAYS)
with session() as con:
    alerts = df(con, '''
        SELECT a.ref_instrument_id, a.rule_name, COALESCE(a.direction, '') AS direction,
               a.trigger_value, a.threshold, a.ts AS alert_ts, a.run_id,
               i.symbol, i.name, i.asset_type, i.currency,
               e.sentiment, e.confidence, e.news_count, e.news_used,
               e.explanation, e.model AS llm_model
        FROM sig_alerts a
        LEFT JOIN ref_instruments i USING (ref_instrument_id)
        LEFT JOIN sig_alert_explanations e
          ON  a.ref_instrument_id = e.ref_instrument_id
          AND a.rule_name         = e.rule_name
          AND COALESCE(a.direction, '') = e.direction
          AND a.ts                = e.ts
        WHERE a.ts >= ?
        ORDER BY a.ts DESC, a.ref_instrument_id
    ''', [since])

print(f'Alerts in letzten {LOOKBACK_DAYS} Tagen: {len(alerts):,}')
if not alerts.empty:
    coverage = 100.0 * alerts['sentiment'].notna().sum() / len(alerts)
    print(f'LLM-Coverage (mit explanation): {coverage:.1f}%')
alerts.head(10)

## 2. Forward-Returns berechnen

Für jedes Alert: für jedes `n` in `FORWARD_WINDOWS` der `n`-te Trading-Tag nach `alert_ts` und der `close` dort vs. `close` am Alert-Tag. Trading-Tage = Tage mit Quote in `mkt_quotes_daily` für das jeweilige Instrument.

In [ ]:
if alerts.empty:
    print('Keine Alerts — Notebook-Rest übersprungen.')
else:
    instruments = alerts['ref_instrument_id'].unique().tolist()
    max_fwd = max(FORWARD_WINDOWS)
    quote_since = since - timedelta(days=5)  # Puffer für alert_ts-close
    quote_until = date.today() + timedelta(days=2)
    with session() as con:
        quotes = df(con, f'''
            WITH ranked AS (
                SELECT ref_instrument_id, ts, close, source,
                       ROW_NUMBER() OVER (
                           PARTITION BY ref_instrument_id, ts
                           ORDER BY CASE source WHEN 'ib' THEN 1 WHEN 'yfinance' THEN 2 ELSE 9 END
                       ) AS rk
                FROM mkt_quotes_daily
                WHERE ts BETWEEN ? AND ?
                  AND ref_instrument_id IN ({','.join(['?']*len(instruments))})
            )
            SELECT ref_instrument_id, ts, close FROM ranked WHERE rk = 1
            ORDER BY ref_instrument_id, ts
        ''', [quote_since, quote_until, *instruments])
    print(f'Forward-Quote-Punkte: {len(quotes):,} für {len(instruments)} Instrumente')
    quotes.head()

In [ ]:
if not alerts.empty:
    # Pro Instrument eine sortierte Quote-Reihe.
    quotes['ts'] = pd.to_datetime(quotes['ts'])
    alerts['alert_ts'] = pd.to_datetime(alerts['alert_ts'])

    def fwd_returns(row, qmap, windows):
        q = qmap.get(row['ref_instrument_id'])
        if q is None or q.empty:
            return {f'ret_{n}d': np.nan for n in windows}
        # Trading-Tage des Instruments, sortiert.
        future = q[q['ts'] >= row['alert_ts']].reset_index(drop=True)
        if future.empty:
            return {f'ret_{n}d': np.nan for n in windows}
        base = future.iloc[0]['close']
        out = {}
        for n in windows:
            if n < len(future):
                out[f'ret_{n}d'] = 100.0 * (future.iloc[n]['close'] / base - 1) if base else np.nan
            else:
                out[f'ret_{n}d'] = np.nan
        return out

    qmap = {ri: g.sort_values('ts') for ri, g in quotes.groupby('ref_instrument_id')}
    ret_records = alerts.apply(lambda r: fwd_returns(r, qmap, FORWARD_WINDOWS), axis=1, result_type='expand')
    alerts_aug = pd.concat([alerts, ret_records], axis=1)

    # 'success' = Forward-Return in der Richtung des Alerts. direction='down' -> negative Returns sind 'right'.
    def dir_sign(d):
        return 1 if d in ('up', 'golden') else (-1 if d in ('down', 'death') else 0)
    alerts_aug['dir_sign'] = alerts_aug['direction'].map(dir_sign)

    for n in FORWARD_WINDOWS:
        alerts_aug[f'hit_{n}d'] = np.where(
            (alerts_aug['dir_sign'] != 0) & alerts_aug[f'ret_{n}d'].notna(),
            np.sign(alerts_aug[f'ret_{n}d']) == alerts_aug['dir_sign'],
            np.nan)
    print('Augmented columns:', [c for c in alerts_aug.columns if c.startswith(('ret_', 'hit_'))])
    alerts_aug[['alert_ts', 'symbol', 'rule_name', 'direction',
                 *[f'ret_{n}d' for n in FORWARD_WINDOWS]]].head(15)

## 3. Hit-Rate pro Regel + Richtung

Direction-Sign muss bekannt sein (`up`/`down`/`golden`/`death`); ohne Direction wird nur Mean-Return reported, keine Hit-Rate.

In [ ]:
if not alerts.empty:
    agg = (alerts_aug
        .groupby(['rule_name', 'direction'], dropna=False)
        .agg(n=('ref_instrument_id', 'count'),
             **{f'mean_ret_{n}d': (f'ret_{n}d', 'mean') for n in FORWARD_WINDOWS},
             **{f'hit_rate_{n}d': (f'hit_{n}d', 'mean') for n in FORWARD_WINDOWS})
        .reset_index()
        .sort_values(['n'], ascending=False))
    # Hit-Rate als %-Werte.
    for n in FORWARD_WINDOWS:
        agg[f'hit_rate_{n}d'] = 100.0 * agg[f'hit_rate_{n}d']
    print(f'Regeln mit >= {MIN_ALERTS_PER_RULE} Alerts:')
    agg[agg['n'] >= MIN_ALERTS_PER_RULE]

In [ ]:
if not alerts.empty:
    plottable = agg[agg['n'] >= MIN_ALERTS_PER_RULE].copy()
    if not plottable.empty:
        plottable['label'] = plottable.apply(
            lambda r: f"{r['rule_name']} ({r['direction'] or '-'}, n={r['n']})", axis=1)
        fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(plottable))))
        width = 0.25
        y = np.arange(len(plottable))
        for i, n in enumerate(FORWARD_WINDOWS):
            ax.barh(y + (i - 1) * width, plottable[f'hit_rate_{n}d'],
                    height=width, label=f'+{n}d')
        ax.set_yticks(y)
        ax.set_yticklabels(plottable['label'])
        hline(ax, 50, label='50% baseline', color=COLORS['neutral'])
        ax.set_xlabel('Hit Rate (%)')
        ax.set_title(f'Hit-Rate pro Regel × Forward-Window (letzte {LOOKBACK_DAYS} Tage)')
        ax.legend(title='Forward'); plt.tight_layout(); plt.show()
    else:
        print(f'Keine Regel hat ≥ {MIN_ALERTS_PER_RULE} Alerts — Plot übersprungen.')

## 4. LLM-Sentiment vs. tatsächlicher Forward-Return

In [ ]:
if not alerts.empty:
    sent = alerts_aug.dropna(subset=['sentiment'])
    if sent.empty:
        print('Keine Alerts mit LLM-Sentiment — alert_explainer nie gelaufen?')
    else:
        sa = (sent.groupby('sentiment')
                  .agg(n=('ref_instrument_id', 'count'),
                       avg_news_used=('news_used', 'mean'),
                       avg_conf=('confidence', 'mean'),
                       **{f'mean_ret_{n}d': (f'ret_{n}d', 'mean') for n in FORWARD_WINDOWS})
                  .reset_index())
        print('Forward-Return pro LLM-Sentiment:')
        sa

In [ ]:
if not alerts.empty and 'sa' in dir() and not sent.empty:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sentiments = sa['sentiment'].tolist()
    x = np.arange(len(sentiments))
    width = 0.25
    for i, n in enumerate(FORWARD_WINDOWS):
        ax.bar(x + (i - 1) * width, sa[f'mean_ret_{n}d'],
               width=width, label=f'+{n}d')
    hline(ax, 0, label='', color=COLORS['neutral'])
    ax.set_xticks(x); ax.set_xticklabels(sentiments)
    ax.set_ylabel('Mean Forward Return (%)')
    ax.set_title('LLM-Sentiment → tatsächlicher Forward-Return')
    ax.legend(title='Forward'); plt.tight_layout(); plt.show()

## 5. Mit-News vs. Ohne-News

Hat das LLM für ein Alert Nachrichten genutzt (`news_used > 0`)? Falls ja — sind diese Alerts qualitativ anders?

In [ ]:
if not alerts.empty:
    aug = alerts_aug.copy()
    aug['has_news'] = (aug['news_used'].fillna(0) > 0)
    nb = (aug.groupby('has_news')
             .agg(n=('ref_instrument_id', 'count'),
                  **{f'mean_ret_{n}d': (f'ret_{n}d', 'mean') for n in FORWARD_WINDOWS},
                  **{f'hit_rate_{n}d': (f'hit_{n}d', 'mean') for n in FORWARD_WINDOWS})
             .reset_index())
    for n in FORWARD_WINDOWS:
        nb[f'hit_rate_{n}d'] = 100.0 * nb[f'hit_rate_{n}d']
    nb

## 6. Raw-Tabelle — alle Alerts + Forward-Returns

Zum manuellen Drill-Down (Excel-Export, Slack-Share, etc.).

In [ ]:
if not alerts.empty:
    cols = ['alert_ts', 'symbol', 'rule_name', 'direction',
            'trigger_value', 'threshold', 'sentiment', 'news_used',
            *[f'ret_{n}d' for n in FORWARD_WINDOWS]]
    alerts_aug[cols].sort_values('alert_ts', ascending=False)